#  Toxic Comment Data Loader

prepares and cleans the Jigsaw toxic comment dataset for model training.
- cleans HTML, emojis, whitespace, and URLs
- balances toxic vs non-toxic samples
- logs key values
- returns a balanced and cleaned DataFrame

If you haven't already, download the data using the terminal:
```
> python dataset/download_dataset.py
```

In [1]:
import pandas as pd
import os
import re
import html
import emoji

TOXIC_LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


##  Step 1: Text Cleaning Function

In [2]:
def clean_comment(text):
    text = str(text)
    text = html.unescape(text) 
    # text = emoji.replace_emoji(text, replace=' [EMOJI] ')  # normalize emoji
    text = emoji.demojize(text) 
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = text.replace("  ", " ").replace("   ", " ").replace("	", " ")
    text = re.sub(r"http\S+", "[URL]", text)  
    text = re.sub(r"\s+", " ", text) 
    return text.strip().strip('"').strip("'")


## Step 2: Prepare Balanced Dataset

In [3]:
def prepare_balanced_dataset(filepath, toxic_labels=TOXIC_LABELS, random_seed=42):
    df = pd.read_csv(filepath)

    print(f"📦 Original dataset shape: {df.shape}")

    # drop missing labels
    df = df.dropna(subset=toxic_labels).copy()

    # clean text
    df['comment_text'] = df['comment_text'].apply(clean_comment)

    # log content analysis
    emoji_count = df['comment_text'].str.contains(r'\[EMOJI\]', regex=True).sum()
    html_count = df['comment_text'].str.contains(r'&[a-z]+;', regex=True).sum()
    url_count = df['comment_text'].str.contains(r'http\S+', regex=True).sum()

    print(f" Comments with [EMOJI]: {emoji_count}")
    print(f" Comments with HTML entities: {html_count}")
    
    print(f" Comments with URLs: {url_count}")

    # balance data
    df["num_labels"] = df[toxic_labels].sum(axis=1)
    toxic_df = df[df["num_labels"] > 0]
    non_toxic_df = df[df["num_labels"] == 0].sample(n=len(toxic_df), random_state=random_seed)

    print(f"✅ Toxic comments: {len(toxic_df)}")
    print(f"✅ Non-toxic sampled: {len(non_toxic_df)}")

    balanced_df = pd.concat([toxic_df, non_toxic_df]).sample(frac=1, random_state=random_seed).reset_index(drop=True)
    print(f"🎯 Final balanced dataset: {len(balanced_df)} rows")

    return balanced_df


## Step 3: Load and Run

In [11]:
# Load and balance dataset
# update if necessary 
import zipfile
from pathlib import Path
import pd

folder_path = "../dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge"

def load_balanced_dataset(
    folder_path: str,
    csv_name: str = "train.csv"
) -> pd.DataFrame:
    """
    Load the balanced dataset from a CSV file, unzipping if a zip archive exists.

    Args:
        folder: Path to the directory containing the dataset.
        csv_name: Name of the CSV file to load.

    Returns:
        A pandas DataFrame of the balanced dataset.
    """
    folder_path = Path(folder)
    csv_path = folder_path / csv_name
    zip_path = folder_path / f"{csv_name}.zip"

    if csv_path.exists():
        return prepare_balanced_dataset(str(csv_path))
    
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extract(csv_name, path=folder_path)
    print("Unzip complete.")
    
    if not csv_path.exists():
        raise FileNotFoundError(f"File not found: {csv_path}")

    return prepare_balanced_dataset(str(csv_path))

balanced_df = load_balanced_dataset(folder_path)


📦 Original dataset shape: (159571, 8)
 Comments with [EMOJI]: 0
 Comments with HTML entities: 76
 Comments with URLs: 0
✅ Toxic comments: 16225
✅ Non-toxic sampled: 16225
🎯 Final balanced dataset: 32450 rows


In [12]:
balanced_df["comment_text"][0]

'You are an old cougar! You are an old cougar!'